# Lesson 2.2 - Functions and Advanced Parameters

**Level: Intermediate**

## Learning Objectives

By the end of this lesson, you will be able to:

- Write reusable PowerShell functions
- Add parameters and validation to a function
- Mark a script as an advanced function with comment-based help

**Project file(s) produced:** `scripts/lesson-2.2-functions-advanced-parameters.ps1`

## Concepts

A **function** is a named block of logic that you can call again and again. Functions help you avoid copy/paste automation, which makes scripts easier to test and change.

An **advanced function** behaves more like a built-in cmdlet. It can accept parameters with validation, support `-WhatIf`, and use PowerShell help metadata.

Reference: [PowerShell Documentation](https://learn.microsoft.com/en-us/powershell/)

In [ ]:
from pathlib import Path
import shutil
import subprocess
import textwrap

lesson_dir = Path('study-plan/topic-powershell/phase-2-scripting-mechanics')
artifact_dir = lesson_dir / 'artifacts'
artifact_dir.mkdir(parents=True, exist_ok=True)

def run_powershell(script_text: str, script_name: str) -> int:
    script_path = lesson_dir / script_name
    script_path.write_text(script_text, encoding='utf-8')
    executable = shutil.which('pwsh') or shutil.which('powershell')
    if executable:
        result = subprocess.run([executable, '-NoLogo', '-NoProfile', '-ExecutionPolicy', 'Bypass', '-File', str(script_path)], capture_output=True, text=True)
        print(result.stdout)
        if result.returncode != 0:
            print(result.stderr)
        return result.returncode
    print(f'{script_path.name} written. Run it in PowerShell to see the live output.')
    return 0

## Code-Along: Step 1 - Build a reusable function

This step writes a function that reads a file and returns simple metadata. Reusable functions are the core of a personal PowerShell module later in the plan.

In [ ]:
script = textwrap.dedent(r'''
function Get-FileMetrics {
    param(
        [Parameter(Mandatory)]
        [string]$Path
    )

    $item = Get-Item -Path $Path
    [pscustomobject]@{
        Name      = $item.Name
        SizeBytes = $item.Length
        Modified  = $item.LastWriteTime
    }
}

Get-FileMetrics -Path $PSCommandPath | Format-List
''')
run_powershell(script, 'lesson-2.2-file-metrics.ps1')

## Code-Along: Step 2 - Add validation and help

This version adds parameter validation and comment-based help so the function behaves more like a polished cmdlet. That makes the script easier to reuse in a larger automation workflow.

In [ ]:
script = textwrap.dedent(r'''
function New-FileInventoryReport {
    [CmdletBinding(SupportsShouldProcess)]
    param(
        [Parameter(Mandatory)]
        [ValidateNotNullOrEmpty()]
        [string]$InputPath,

        [Parameter()]
        [string]$OutputPath = (Join-Path $PSScriptRoot 'artifacts/file-inventory.txt')
    )

    if ($PSCmdlet.ShouldProcess($InputPath, 'Create file inventory report')) {
        Get-ChildItem -Path $InputPath | Select-Object Name, Length, LastWriteTime | Format-Table -AutoSize | Out-String | Set-Content -Path $OutputPath
        Get-Content -Path $OutputPath
    }
}

New-FileInventoryReport -InputPath $PSScriptRoot
''')
run_powershell(script, 'lesson-2.2-advanced-function.ps1')

## Exercise

Add one more parameter to the advanced function so the caller can filter by file extension before the report is written.

**Your task:**
1. Add a new validated parameter.
2. Use it to filter the file list.
3. Re-run the report and confirm the output changes.

**Expected result:** a reusable function that is a little closer to a production script.

In [ ]:
# TODO: add a file extension filter to the advanced function
exercise_script = textwrap.dedent(r'''
Write-Host "TODO: add one more parameter and use it in the report"
''')
run_powershell(exercise_script, 'lesson-2.2-exercise.ps1')

## Key Takeaways

- Functions let you package automation into reusable units.
- Advanced functions behave more like cmdlets and are easier to standardize.
- Parameter validation is a simple way to catch mistakes early.

---

**Next:** [Lesson 2.3 - Errors, Formatting, and JSON](./lesson-2.3-errors-formatting-json.ipynb)